In [1]:
import pandas as pd
import time
import sqlite3
from nba_api.stats.endpoints import LeagueDashPtStats

In [2]:
season = "2024-25"

In [3]:
from nba_api.stats.endpoints import LeagueGameFinder

# Set parameters for the desired season and team
gamefinder = LeagueGameFinder(season_nullable=season,season_type_nullable="Regular Season")

# Get games as a DataFrame
games = gamefinder.get_data_frames()[0]

# Optional: Filter to only include date, home and away teams, and matchup info
schedule = pd.DataFrame(games[['GAME_DATE']])

# Convert game date to a readable format if needed
schedule['GAME_DATE'] = pd.to_datetime(schedule['GAME_DATE']).dt.strftime('%m/%d/%Y')

In [4]:
dates = schedule.drop_duplicates()

In [5]:
from nba_api.stats.endpoints import CommonAllPlayers
import pandas as pd

# Get all players for the 2023-24 season
players_data = CommonAllPlayers(is_only_current_season=1, league_id='00', season='2024-25')
players_df = players_data.get_data_frames()[0]

# Extract relevant player IDs and names
player_ids = players_df['PERSON_ID'].tolist()

# TRACKING: Catch and Shoot

In [6]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM catchshoot_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [7]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['CatchShoot']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 30 dates left to process.
Batch completed. 10 dates remaining.
Starting batch with 10 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [8]:
catchshoot_df = tracking_df_total

In [9]:
catchshoot_df['SEASON_YEAR']=season
catchshoot_df['id'] = catchshoot_df['PLAYER_ID'].astype(str) +"_"+catchshoot_df['GAME_DATE'].astype(str)+"_"+catchshoot_df['Tracking'].astype(str)

In [10]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "catchshoot_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    catchshoot_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = catchshoot_df[~catchshoot_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'catchshoot_player_data' already exists. Checking for new records...
Inserted 5160 new records into 'catchshoot_player_data'.


# TRACKING: Pullup

In [11]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM pullup_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [12]:
#unique_dates

In [13]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['PullUpShot']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 30 dates left to process.
Batch completed. 10 dates remaining.
Starting batch with 10 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [14]:
pullup_df = tracking_df_total

In [15]:
pullup_df

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,MIN,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG_PCT,PULL_UP_PTS,PULL_UP_FG3M,PULL_UP_FG3A,PULL_UP_FG3_PCT,PULL_UP_EFG_PCT,GAME_DATE,Tracking
0,1630639,A.J. Lawson,1610612761,TOR,1,0,1,21.3,0.0,0.0,NaN,0.0,0.0,0.0,NaN,NaN,04/13/2025,PullUpShot
1,1631260,AJ Green,1610612749,MIL,1,1,0,18.9,1.0,4.0,0.250,3.0,1.0,3.0,0.333,0.375,04/13/2025,PullUpShot
2,1642358,AJ Johnson,1610612764,WAS,1,1,0,48.0,1.0,3.0,0.333,3.0,1.0,2.0,0.500,0.500,04/13/2025,PullUpShot
3,203932,Aaron Gordon,1610612743,DEN,1,1,0,26.4,0.0,2.0,0.000,0.0,0.0,2.0,0.000,0.000,04/13/2025,PullUpShot
4,1628988,Aaron Holiday,1610612745,HOU,1,0,1,9.4,1.0,2.0,0.500,3.0,1.0,2.0,0.500,0.750,04/13/2025,PullUpShot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5155,1631111,Wendell Moore Jr.,1610612766,CHA,1,1,0,8.6,0.0,1.0,0.000,0.0,NaN,NaN,NaN,0.000,03/14/2025,PullUpShot
5156,1642258,Zaccharie Risacher,1610612737,ATL,1,0,1,36.5,0.0,0.0,NaN,0.0,0.0,0.0,NaN,NaN,03/14/2025,PullUpShot
5157,1641744,Zach Edey,1610612763,MEM,1,0,1,10.8,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,03/14/2025,PullUpShot
5158,203897,Zach LaVine,1610612758,SAC,1,0,1,36.2,3.0,8.0,0.375,7.0,1.0,5.0,0.200,0.438,03/14/2025,PullUpShot


In [16]:
pullup_df['SEASON_YEAR']=season
pullup_df['id'] = pullup_df['PLAYER_ID'].astype(str) +"_"+pullup_df['GAME_DATE'].astype(str)+"_"+pullup_df['Tracking'].astype(str)

In [17]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "pullup_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    pullup_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = pullup_df[~pullup_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'pullup_player_data' already exists. Checking for new records...
Inserted 5160 new records into 'pullup_player_data'.


# TRACKING: Passing

In [18]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM passing_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [19]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['Passing']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 32 dates left to process.
Batch completed. 12 dates remaining.
Starting batch with 12 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [20]:
passing_df = tracking_df_total

In [21]:
passing_df['SEASON_YEAR']=season
passing_df['id'] = passing_df['PLAYER_ID'].astype(str) +"_"+passing_df['GAME_DATE'].astype(str)+"_"+passing_df['Tracking'].astype(str)

In [22]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "passing_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    passing_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = passing_df[~passing_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'passing_data' already exists. Checking for new records...
Inserted 5431 new records into 'passing_data'.


# TRACKING: Rebounding

In [23]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM rebounding_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [24]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['Rebounding']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 32 dates left to process.
Batch completed. 12 dates remaining.
Starting batch with 12 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [25]:
rebounding_df = tracking_df_total

In [26]:
rebounding_df['SEASON_YEAR']=season
rebounding_df['id'] = rebounding_df['PLAYER_ID'].astype(str) +"_"+rebounding_df['GAME_DATE'].astype(str)+"_"+rebounding_df['Tracking'].astype(str)

In [27]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "rebounding_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    rebounding_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = rebounding_df[~rebounding_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'rebounding_player_data' already exists. Checking for new records...
Inserted 4844 new records into 'rebounding_player_data'.


# TRACKING: Drives

In [28]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM drives_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [29]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['Drives']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [30]:
drives_df = tracking_df_total

In [31]:
drives_df['SEASON_YEAR']=season
drives_df['id'] = drives_df['PLAYER_ID'].astype(str) +"_"+drives_df['GAME_DATE'].astype(str)+"_"+drives_df['Tracking'].astype(str)

In [32]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "drives_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()


# Step 2: Create the table if it doesn't exist
if not table_exists:
    drives_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = drives_df[~drives_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'drives_player_data' already exists. Checking for new records...
Inserted 5319 new records into 'drives_player_data'.


# TRACKING: Speed and Distance

In [33]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM speeddistance_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [34]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['SpeedDistance']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [35]:
speeddistance_df = tracking_df_total

In [36]:
speeddistance_df['SEASON_YEAR']=season
speeddistance_df['id'] = speeddistance_df['PLAYER_ID'].astype(str) +"_"+speeddistance_df['GAME_DATE'].astype(str)+"_"+speeddistance_df['Tracking'].astype(str)

In [37]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "speeddistance_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [38]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    speeddistance_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = speeddistance_df[~speeddistance_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'speeddistance_player_data' already exists. Checking for new records...
Inserted 5319 new records into 'speeddistance_player_data'.


# TRACKING: PostTouch

In [39]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM posttouch_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [40]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['PostTouch']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame

# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [41]:
posttouch_df = tracking_df_total

In [42]:
posttouch_df['SEASON_YEAR']=season
posttouch_df['id'] = posttouch_df['PLAYER_ID'].astype(str) +"_"+posttouch_df['GAME_DATE'].astype(str)+"_"+posttouch_df['Tracking'].astype(str)

In [43]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "posttouch_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    posttouch_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = posttouch_df[~posttouch_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'posttouch_player_data' already exists. Checking for new records...
Inserted 5319 new records into 'posttouch_player_data'.


In [44]:
#missed_dates = ['03/22/2024','12/15/2023']

In [45]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM painttouch_player_data", conn)

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [46]:
# Initialize an empty DataFrame to collect all tracking data
tracking_df_total = pd.DataFrame()

# Filter options for tracking
filter_options = ['PaintTouch']

# Initial list of dates to process
dates_new = unique_dates[['GAME_DATE']].copy()  # Reference to the main dates DataFrame
#dates_new = missed_dates
# Dictionary to log failed dates and filter options
failed_log = {}

# Loop to fetch data until all dates are processed
while not dates_new.empty:
    all_data = []  # Temporary storage for each iteration
    print(f"Starting batch with {len(dates_new)} dates left to process.")

    # Process the first 20 dates in dates_new
    batch_dates = dates_new['GAME_DATE'].head(20).tolist()  # Get the first 20 dates as a list

    # Loop through each date and filter option
    for date in batch_dates:
        for tracking in filter_options:
            attempt = 0
            success = False

            while attempt < 3 and not success:
                try:
                    shot_data = LeagueDashPtStats(
                        season=season,
                        per_mode_simple='PerGame',
                        pt_measure_type=tracking,
                        date_from_nullable=date,
                        date_to_nullable=date,
                        player_or_team='Player'
                    )
                    temp_df = shot_data.get_data_frames()[0]
                    temp_df['GAME_DATE'] = date
                    temp_df['Tracking'] = tracking
                    
                    all_data.append(temp_df)
                    success = True
                    
                    time.sleep(4)  # Wait time between successful requests
                except Exception as e:
                    attempt += 1
                    print(f"Attempt {attempt} failed for date {date} and tracking {tracking}: {e}")
                    time.sleep(5 * attempt)

            # Log failures if unsuccessful after all attempts
            if not success:
                print(f"Failed to retrieve data for date {date} and tracking {tracking} after 3 attempts.")
                failed_log.setdefault(date, []).append(tracking)  # Log the failed filter for the date

    # Combine current batch into a single DataFrame
    tracking_df = pd.concat(all_data, ignore_index=True)
    
    # Append new data to the total DataFrame
    tracking_df_total = pd.concat([tracking_df_total, tracking_df], ignore_index=True)

    # Remove processed dates from `dates_new`
    dates_new = dates_new[~dates_new['GAME_DATE'].isin(batch_dates)]

    # Display the progress
    print(f"Batch completed. {len(dates_new)} dates remaining.")
    time.sleep(15)

# Display or process `tracking_df_total`
print("Data retrieval complete.")

# Log the failed dates and tracking options
if failed_log:
    print("Failed dates and their tracking options:")
    for date, filters in failed_log.items():
        print(f"Date: {date}, Failed filters: {filters}")
else:
    print("All dates processed successfully.")

Starting batch with 31 dates left to process.
Batch completed. 11 dates remaining.
Starting batch with 11 dates left to process.
Batch completed. 0 dates remaining.
Data retrieval complete.
All dates processed successfully.


In [47]:
painttouch_df = tracking_df_total

In [48]:
painttouch_df['SEASON_YEAR']=season
painttouch_df['id'] = painttouch_df['PLAYER_ID'].astype(str) +"_"+painttouch_df['GAME_DATE'].astype(str)+"_"+painttouch_df['Tracking'].astype(str)

In [49]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "painttouch_player_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    painttouch_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = painttouch_df[~painttouch_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'painttouch_player_data' already exists. Checking for new records...
Inserted 5319 new records into 'painttouch_player_data'.


In [50]:
from datetime import datetime

In [51]:
#existing_dates['GAME_DATE'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')

In [52]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM gamelogs", conn)

existing_dates['GAME_DATE'] = pd.to_datetime(existing_dates['GAME_DATE']).dt.strftime('%m/%d/%Y')

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [53]:
unique_dates['GAME_DATE'].iloc[-1]

'03/14/2025'

In [54]:
failed_player_ids = [1626164, 1642402, 203497, 1641789, 1630643, 1631127, 1630200, 1641798, 1630231, 1642263, 1641733, 1642281, 1630649, 1629020, 1630592]

In [55]:
from nba_api.stats.endpoints import PlayerGameLogs
import time
all_gamelog_data = []
# Loop through each player ID to fetch shot data for the entire season
for player_id in player_ids:  # Using [:10] as a subset for testing
    attempt = 0
    success = False
    
    while attempt < 3 and not success:
        try:
            # Fetch shot chart details for the player across the entire season
            gamelog = PlayerGameLogs(player_id_nullable=player_id, season_nullable=season, season_type_nullable='Regular Season' , date_from_nullable =unique_dates['GAME_DATE'].iloc[-1],date_to_nullable =unique_dates['GAME_DATE'].iloc[0] )
            # Convert the data to a DataFrame
            gamelog_df = gamelog.get_data_frames()[0]
            
            # Append the DataFrame to the list
            all_gamelog_data.append(gamelog_df)
            success = True
            
            # Respect API rate limits by adding a delay
            time.sleep(2)
        
        except Exception as e:
            attempt += 1
            print(f"Attempt {attempt} failed for player {player_id}: {e}")
            time.sleep(5* attempt)

# Combine all shot details into one DataFrame
#all_shots_df = pd.concat(all_shot_data, ignore_index=True)
gamelog_data_df = pd.concat(all_gamelog_data, ignore_index=True)

Attempt 1 failed for player 1626167: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)


In [56]:
#gamelog_data_df

In [57]:
gamelog_data_df['id'] = gamelog_data_df['PLAYER_ID'].astype(str) +"_"+gamelog_data_df['GAME_DATE'].astype(str)+"_"+gamelog_data_df['TEAM_ID'].astype(str)
#gamelog_data_df['SEASON_YEAR'] = season_year
gamelog_data_df['PLAYER_NAME'].nunique()

490

In [58]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gamelogs"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gamelog_data_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gamelog_data_df[~gamelog_data_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gamelogs' already exists. Checking for new records...
Inserted 5160 new records into 'gamelogs'.
